# Lab 4 · เตรียมข้อมูลและ fine-tune ด้วย LoRA
**สัปดาห์ 3 · ช่วงบ่าย (60 นาที)**

| ขั้น | ทำอะไร | เห็นอะไร |
|---|---|---|
| 4.1 | แปลงข้อมูลเป็นบทสนทนา (chat template) 100 train / 20 eval | ส่วนไหนเป็น **input** ส่วนไหนเป็น **label** ทีละ token |
| 4.2 | ใส่ LoRA rank 16 ที่ q, k, v, o | จำนวนพารามิเตอร์ที่เทรนจริง เทียบกับที่คำนวณด้วยมือ |
| 4.3 | เทรน ≈ 1–3 นาที | ตารางและกราฟ train / eval loss |
| 4.4 | เปิดชุด test 20 ข้อครั้งแรก เทียบ 3 แนวทาง | ตารางคะแนน + ค่า p ว่าต่างจริงหรือบังเอิญ |

ใช้ไลบรารีมาตรฐานของ Hugging Face (transformers + peft) — ติดตั้งราว 1 นาที **ไม่ต้อง restart**
> **ก่อนรัน:** ปิด runtime ของ Lab 3 ก่อน (Runtime → Disconnect and delete runtime) · Runtime → Change runtime type → **T4 GPU** · ถ้ามีไฟล์ `best_prompts.json` จาก Lab 3 ให้ลากไปวางในแผง Files (ไอคอนโฟลเดอร์ซ้ายมือ)

In [ ]:
!pip install -q -U "transformers>=4.51,<5" "peft>=0.12" accelerate datasets
import torch, json, re, random, math, glob, contextlib
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
import warnings, transformers; warnings.filterwarnings("ignore"); transformers.utils.logging.set_verbosity_error()   # ซ่อน warning ที่ไม่เกี่ยวกับบทเรียน
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, os, urllib.request
FONT = "Sarabun-Regular.ttf"                       # ฟอนต์ไทยสำหรับกราฟ (ฟอนต์เริ่มต้นของ Colab ไม่มีอักษรไทย)
if not os.path.exists(FONT):
    try: urllib.request.urlretrieve("https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", FONT)
    except Exception as e: print("ดาวน์โหลดฟอนต์ไม่ได้ (กราฟจะไม่แสดงอักษรไทย):", e)
if os.path.exists(FONT):
    fm.fontManager.addfont(FONT); plt.rcParams["font.family"] = [fm.FontProperties(fname=FONT).get_name(), "DejaVu Sans"]   # ตัวหลังสำรองสำหรับสัญลักษณ์ที่ Sarabun ไม่มี
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่พบ GPU — เปลี่ยน runtime เป็น T4")

In [ ]:
MODEL = "scb10x/llama3.2-typhoon2-1b-instruct"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32   # โมเดล 1B แบบ fp16 ใช้ ≈ 2.5 GB — ไม่ต้อง 4-bit
tok = AutoTokenizer.from_pretrained(MODEL)
EOT = tok.convert_tokens_to_ids("<|eot_id|>")      # token จบคำตอบของ Llama 3 (ใช้ทั้งตอนเทรนและตอน generate)
if tok.pad_token is None:
    tok.pad_token = "<|eot_id|>"                   # token ไว้เติมความยาวใน batch ให้เท่ากัน
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=DTYPE).to("cuda" if torch.cuda.is_available() else "cpu").eval()
DEVICE = next(model.parameters()).device
print("โหลดเสร็จ:", model.config.num_hidden_layers, "ชั้น")

In [ ]:
#@title ชุดข้อมูลของสัปดาห์ 3 (ฝังไว้ในไฟล์ ไม่ต้องอัปโหลด · โค้ดถูกซ่อน ดับเบิลคลิกถ้าอยากดู) { display-mode: "form" }
# extract  = โจทย์ (ข) สกัดชื่อสินค้าและราคา  |  classify = โจทย์ (ค) จัดหมวดหมู่คำร้องเรียน 4 หมวด
# dev/test เขียนด้วยมือ · train 120 ข้อสร้างจากแม่แบบ (ข้อมูลสังเคราะห์ ใช้เพื่อการเรียนเท่านั้น)
import json
DATA = json.loads(r'''{"extract": {"dev": [{"text": "หูฟังไร้สาย SoundPro X2 ราคา 1,290 บาท ส่งฟรี", "product": "หูฟังไร้สาย SoundPro X2", "price": 1290}, {"text": "ซื้อกระติกน้ำเก็บความเย็น Hydro 750ml มาในราคา 590 บาท คุ้มมาก", "product": "กระติกน้ำเก็บความเย็น Hydro 750ml", "price": 590}, {"text": "โปรวันนี้! เมาส์ไร้สาย Logi M240 เหลือเพียง 450 บาท", "product": "เมาส์ไร้สาย Logi M240", "price": 450}, {"text": "แป้นพิมพ์ mechanical KeyMaster 87 ตัวนี้ 2,490 บาท มีของแถมเป็นสายถัก", "product": "แป้นพิมพ์ mechanical KeyMaster 87", "price": 2490}, {"text": "เก้าอี้สำนักงาน ErgoFlex ราคา ฿3,990 รับประกัน 2 ปี", "product": "เก้าอี้สำนักงาน ErgoFlex", "price": 3990}, {"text": "หม้อทอดไร้น้ำมัน AirCook 5L ลดเหลือ 1,790 บาท จาก 2,590", "product": "หม้อทอดไร้น้ำมัน AirCook 5L", "price": 1790}, {"text": "ขายกล้อง instant Snapy Mini ราคา 2,150 บาท พร้อมฟิล์ม 1 กล่อง", "product": "กล้อง instant Snapy Mini", "price": 2150}, {"text": "รองเท้าวิ่ง SwiftRun 3 ไซส์ 42 ราคา 2,890 บาท สภาพใหม่", "product": "รองเท้าวิ่ง SwiftRun 3", "price": 2890}, {"text": "โคมไฟตั้งโต๊ะ LumiDesk ปรับแสงได้ 3 ระดับ 690 บาทเท่านั้น", "product": "โคมไฟตั้งโต๊ะ LumiDesk", "price": 690}, {"text": "สายชาร์จ USB-C ยาว 2 เมตร ยี่ห้อ Volt ราคา 199 บาท", "product": "สายชาร์จ USB-C ยาว 2 เมตร ยี่ห้อ Volt", "price": 199}], "test": [{"text": "พัดลมตั้งพื้น CoolBreeze 16 นิ้ว ราคา 1,150 บาท ส่งถึงบ้าน", "product": "พัดลมตั้งพื้น CoolBreeze 16 นิ้ว", "price": 1150}, {"text": "ลำโพงบลูทูธ BoomBox Mini ตัวเล็กเสียงดี 890 บาท", "product": "ลำโพงบลูทูธ BoomBox Mini", "price": 890}, {"text": "นาฬิกาอัจฉริยะ FitTime S ราคาโปร 2,990 บาท จากปกติ 3,990", "product": "นาฬิกาอัจฉริยะ FitTime S", "price": 2990}, {"text": "เครื่องชงกาแฟแคปซูล Brewly One ราคา ฿4,590 แถมแคปซูล 20 ชิ้น", "product": "เครื่องชงกาแฟแคปซูล Brewly One", "price": 4590}, {"text": "กระเป๋าเป้ TrailPack 30L กันน้ำ ราคา 1,490 บาท", "product": "กระเป๋าเป้ TrailPack 30L", "price": 1490}, {"text": "แท็บเล็ต NotePad 10 รุ่น Wi-Fi 64GB ขาย 6,900 บาท", "product": "แท็บเล็ต NotePad 10 รุ่น Wi-Fi 64GB", "price": 6900}, {"text": "ที่นอนยางพารา DreamSoft 6 ฟุต ราคา 12,500 บาท ส่งฟรีทั่วไทย", "product": "ที่นอนยางพารา DreamSoft 6 ฟุต", "price": 12500}, {"text": "เครื่องดูดฝุ่นไร้สาย DustAway V3 เหลือ 3,290 บาท ถึงสิ้นเดือน", "product": "เครื่องดูดฝุ่นไร้สาย DustAway V3", "price": 3290}, {"text": "จอมอนิเตอร์ ViewMax 27 นิ้ว 144Hz ราคา 5,990 บาท", "product": "จอมอนิเตอร์ ViewMax 27 นิ้ว 144Hz", "price": 5990}, {"text": "หมวกกันน็อก SafeRide รุ่นเต็มใบ 1,850 บาท มีให้เลือก 3 สี", "product": "หมวกกันน็อก SafeRide รุ่นเต็มใบ", "price": 1850}, {"text": "ไมโครโฟน USB CastPro ราคา 1,690 บาท เหมาะกับสตรีมเกม", "product": "ไมโครโฟน USB CastPro", "price": 1690}, {"text": "เตารีดไอน้ำ SteamEase ราคาเพียง 790 บาท", "product": "เตารีดไอน้ำ SteamEase", "price": 790}, {"text": "จักรยานพับได้ FoldGo 20 นิ้ว ราคา 7,900 บาท พร้อมประกัน", "product": "จักรยานพับได้ FoldGo 20 นิ้ว", "price": 7900}, {"text": "เครื่องฟอกอากาศ PureAir 300 ราคา ฿3,490 กรอง PM2.5", "product": "เครื่องฟอกอากาศ PureAir 300", "price": 3490}, {"text": "กระทะเคลือบ NonStick Pro 28 ซม. ราคา 650 บาท", "product": "กระทะเคลือบ NonStick Pro 28 ซม.", "price": 650}, {"text": "ปากกาสไตลัส InkTip 2 ราคา 1,290 บาท ใช้กับแท็บเล็ตทุกรุ่น", "product": "ปากกาสไตลัส InkTip 2", "price": 1290}, {"text": "เสื้อกันฝน RainShield แบบพกพา 350 บาท", "product": "เสื้อกันฝน RainShield แบบพกพา", "price": 350}, {"text": "ตู้เย็นมินิ ChillBox 45L ราคา 3,990 บาท เหมาะกับหอพัก", "product": "ตู้เย็นมินิ ChillBox 45L", "price": 3990}, {"text": "แบตเตอรี่สำรอง PowerGo 20000mAh ราคา 990 บาท ชาร์จเร็ว", "product": "แบตเตอรี่สำรอง PowerGo 20000mAh", "price": 990}, {"text": "โต๊ะทำงานปรับระดับ LiftDesk 120 ซม. ราคา 8,900 บาท", "product": "โต๊ะทำงานปรับระดับ LiftDesk 120 ซม.", "price": 8900}], "train": [{"text": "หูฟังไร้สาย EchoBud ราคาปกติ 890 บาท", "product": "หูฟังไร้สาย EchoBud", "price": 890}, {"text": "ลำโพงพกพา MiniBeat ลดเหลือ 3,490 บาท วันนี้เท่านั้น", "product": "ลำโพงพกพา MiniBeat", "price": 3490}, {"text": "เมาส์เกมมิ่ง ClickX ราคาปกติ 1,490 บาท", "product": "เมาส์เกมมิ่ง ClickX", "price": 1490}, {"text": "คีย์บอร์ดไร้สาย TypeLite ราคาปกติ 2,490 บาท", "product": "คีย์บอร์ดไร้สาย TypeLite", "price": 2490}, {"text": "กระติกน้ำ ThermoCup 500ml ราคาปกติ 350 บาท", "product": "กระติกน้ำ ThermoCup 500ml", "price": 350}, {"text": "หม้อหุงข้าว RiceMate 1.8L ตัวนี้ 199 บาท มีของแถม", "product": "หม้อหุงข้าว RiceMate 1.8L", "price": 199}, {"text": "รีวิว เครื่องปั่น BlendGo ซื้อมา 990 บาท ใช้ดี", "product": "เครื่องปั่น BlendGo", "price": 990}, {"text": "โคมไฟอ่านหนังสือ ReadLite ราคาเพียง 890 บาท ส่งฟรี", "product": "โคมไฟอ่านหนังสือ ReadLite", "price": 890}, {"text": "รีวิว พัดลมพกพา BreezePocket ซื้อมา 2,490 บาท ใช้ดี", "product": "พัดลมพกพา BreezePocket", "price": 2490}, {"text": "กระเป๋าเดินทาง RollEasy 24 นิ้ว ตัวนี้ 3,490 บาท มีของแถม", "product": "กระเป๋าเดินทาง RollEasy 24 นิ้ว", "price": 3490}, {"text": "รองเท้าผ้าใบ UrbanStep ลดเหลือ 1,690 บาท วันนี้เท่านั้น", "product": "รองเท้าผ้าใบ UrbanStep", "price": 1690}, {"text": "เสื้อแจ็กเก็ตกันลม WindGuard ลดเหลือ 890 บาท วันนี้เท่านั้น", "product": "เสื้อแจ็กเก็ตกันลม WindGuard", "price": 890}, {"text": "สั่ง นาฬิกาข้อมือ TimeCore มาแล้ว 2,990 บาท คุ้มมาก", "product": "นาฬิกาข้อมือ TimeCore", "price": 2990}, {"text": "ขาย แว่นกันแดด SunVue ในราคา 199 บาท", "product": "แว่นกันแดด SunVue", "price": 199}, {"text": "หมวกแก๊ป CapOne ราคาปกติ 690 บาท", "product": "หมวกแก๊ป CapOne", "price": 690}, {"text": "โปรพิเศษ แท่นชาร์จไร้สาย ChargePad เหลือ 259 บาท", "product": "แท่นชาร์จไร้สาย ChargePad", "price": 259}, {"text": "โปรพิเศษ สายชาร์จ Lightning FastLink เหลือ 199 บาท", "product": "สายชาร์จ Lightning FastLink", "price": 199}, {"text": "ที่วางโทรศัพท์ HoldUp ราคาปกติ 2,490 บาท", "product": "ที่วางโทรศัพท์ HoldUp", "price": 2490}, {"text": "สั่ง กล้องติดรถ DriveCam HD มาแล้ว 1,690 บาท คุ้มมาก", "product": "กล้องติดรถ DriveCam HD", "price": 1690}, {"text": "เครื่องวัดความดัน HealthCheck ราคาปกติ 1,690 บาท", "product": "เครื่องวัดความดัน HealthCheck", "price": 1690}, {"text": "ไม้เซลฟี่ SnapStick ลดเหลือ 2,190 บาท วันนี้เท่านั้น", "product": "ไม้เซลฟี่ SnapStick", "price": 2190}, {"text": "ขาย เต็นท์ 2 คน CampLite ในราคา 1,490 บาท", "product": "เต็นท์ 2 คน CampLite", "price": 1490}, {"text": "ถุงนอน WarmNest ลดเหลือ 259 บาท วันนี้เท่านั้น", "product": "ถุงนอน WarmNest", "price": 259}, {"text": "ไฟฉาย LED BrightBeam ราคาเพียง 2,490 บาท ส่งฟรี", "product": "ไฟฉาย LED BrightBeam", "price": 2490}, {"text": "สั่ง เขียงไม้ ChopWood มาแล้ว 990 บาท คุ้มมาก", "product": "เขียงไม้ ChopWood", "price": 990}, {"text": "โปรพิเศษ มีดเชฟ SharpEdge 8 นิ้ว เหลือ 5,900 บาท", "product": "มีดเชฟ SharpEdge 8 นิ้ว", "price": 5900}, {"text": "รีวิว ชุดเครื่องนอน SoftDream 6 ฟุต ซื้อมา 1,890 บาท ใช้ดี", "product": "ชุดเครื่องนอน SoftDream 6 ฟุต", "price": 1890}, {"text": "หมอนยางพารา NeckEase ราคาปกติ 1,690 บาท", "product": "หมอนยางพารา NeckEase", "price": 1690}, {"text": "รีวิว พรมเช็ดเท้า StepDry ซื้อมา 1,490 บาท ใช้ดี", "product": "พรมเช็ดเท้า StepDry", "price": 1490}, {"text": "สั่ง ตะกร้าผ้า FoldBin มาแล้ว 3,990 บาท คุ้มมาก", "product": "ตะกร้าผ้า FoldBin", "price": 3990}, {"text": "หูฟังไร้สาย EchoBud ราคาเพียง 3,990 บาท ส่งฟรี", "product": "หูฟังไร้สาย EchoBud", "price": 3990}, {"text": "ลำโพงพกพา MiniBeat ราคา 1,290 บาท", "product": "ลำโพงพกพา MiniBeat", "price": 1290}, {"text": "เมาส์เกมมิ่ง ClickX ราคาปกติ 990 บาท", "product": "เมาส์เกมมิ่ง ClickX", "price": 990}, {"text": "คีย์บอร์ดไร้สาย TypeLite ลดเหลือ 7,900 บาท วันนี้เท่านั้น", "product": "คีย์บอร์ดไร้สาย TypeLite", "price": 7900}, {"text": "รีวิว กระติกน้ำ ThermoCup 500ml ซื้อมา 1,290 บาท ใช้ดี", "product": "กระติกน้ำ ThermoCup 500ml", "price": 1290}, {"text": "หม้อหุงข้าว RiceMate 1.8L ราคาปกติ 3,990 บาท", "product": "หม้อหุงข้าว RiceMate 1.8L", "price": 3990}, {"text": "เครื่องปั่น BlendGo ราคาเพียง 490 บาท ส่งฟรี", "product": "เครื่องปั่น BlendGo", "price": 490}, {"text": "โคมไฟอ่านหนังสือ ReadLite ราคาปกติ 5,900 บาท", "product": "โคมไฟอ่านหนังสือ ReadLite", "price": 5900}, {"text": "โปรพิเศษ พัดลมพกพา BreezePocket เหลือ 990 บาท", "product": "พัดลมพกพา BreezePocket", "price": 990}, {"text": "ขาย กระเป๋าเดินทาง RollEasy 24 นิ้ว ในราคา 490 บาท", "product": "กระเป๋าเดินทาง RollEasy 24 นิ้ว", "price": 490}, {"text": "รองเท้าผ้าใบ UrbanStep ตัวนี้ 2,490 บาท มีของแถม", "product": "รองเท้าผ้าใบ UrbanStep", "price": 2490}, {"text": "เสื้อแจ็กเก็ตกันลม WindGuard ฿350 มีประกัน", "product": "เสื้อแจ็กเก็ตกันลม WindGuard", "price": 350}, {"text": "สั่ง นาฬิกาข้อมือ TimeCore มาแล้ว 350 บาท คุ้มมาก", "product": "นาฬิกาข้อมือ TimeCore", "price": 350}, {"text": "แว่นกันแดด SunVue ราคา 590 บาท", "product": "แว่นกันแดด SunVue", "price": 590}, {"text": "สั่ง หมวกแก๊ป CapOne มาแล้ว 1,190 บาท คุ้มมาก", "product": "หมวกแก๊ป CapOne", "price": 1190}, {"text": "ขาย แท่นชาร์จไร้สาย ChargePad ในราคา 1,890 บาท", "product": "แท่นชาร์จไร้สาย ChargePad", "price": 1890}, {"text": "สายชาร์จ Lightning FastLink ราคาปกติ 259 บาท", "product": "สายชาร์จ Lightning FastLink", "price": 259}, {"text": "ที่วางโทรศัพท์ HoldUp ราคา 4,590 บาท", "product": "ที่วางโทรศัพท์ HoldUp", "price": 4590}, {"text": "กล้องติดรถ DriveCam HD ราคาปกติ 1,690 บาท", "product": "กล้องติดรถ DriveCam HD", "price": 1690}, {"text": "รีวิว เครื่องวัดความดัน HealthCheck ซื้อมา 1,290 บาท ใช้ดี", "product": "เครื่องวัดความดัน HealthCheck", "price": 1290}, {"text": "รีวิว ไม้เซลฟี่ SnapStick ซื้อมา 990 บาท ใช้ดี", "product": "ไม้เซลฟี่ SnapStick", "price": 990}, {"text": "เต็นท์ 2 คน CampLite ราคา 890 บาท", "product": "เต็นท์ 2 คน CampLite", "price": 890}, {"text": "ถุงนอน WarmNest ราคา 1,190 บาท", "product": "ถุงนอน WarmNest", "price": 1190}, {"text": "ขาย ไฟฉาย LED BrightBeam ในราคา 350 บาท", "product": "ไฟฉาย LED BrightBeam", "price": 350}, {"text": "รีวิว เขียงไม้ ChopWood ซื้อมา 4,590 บาท ใช้ดี", "product": "เขียงไม้ ChopWood", "price": 4590}, {"text": "มีดเชฟ SharpEdge 8 นิ้ว ราคาเพียง 259 บาท ส่งฟรี", "product": "มีดเชฟ SharpEdge 8 นิ้ว", "price": 259}, {"text": "โปรพิเศษ ชุดเครื่องนอน SoftDream 6 ฟุต เหลือ 1,890 บาท", "product": "ชุดเครื่องนอน SoftDream 6 ฟุต", "price": 1890}, {"text": "โปรพิเศษ หมอนยางพารา NeckEase เหลือ 4,590 บาท", "product": "หมอนยางพารา NeckEase", "price": 4590}, {"text": "พรมเช็ดเท้า StepDry ราคา 590 บาท", "product": "พรมเช็ดเท้า StepDry", "price": 590}, {"text": "ตะกร้าผ้า FoldBin ฿1,290 มีประกัน", "product": "ตะกร้าผ้า FoldBin", "price": 1290}, {"text": "หูฟังไร้สาย EchoBud ลดเหลือ 1,490 บาท วันนี้เท่านั้น", "product": "หูฟังไร้สาย EchoBud", "price": 1490}, {"text": "สั่ง ลำโพงพกพา MiniBeat มาแล้ว 1,690 บาท คุ้มมาก", "product": "ลำโพงพกพา MiniBeat", "price": 1690}, {"text": "รีวิว เมาส์เกมมิ่ง ClickX ซื้อมา 2,190 บาท ใช้ดี", "product": "เมาส์เกมมิ่ง ClickX", "price": 2190}, {"text": "คีย์บอร์ดไร้สาย TypeLite ราคาปกติ 1,690 บาท", "product": "คีย์บอร์ดไร้สาย TypeLite", "price": 1690}, {"text": "รีวิว กระติกน้ำ ThermoCup 500ml ซื้อมา 7,900 บาท ใช้ดี", "product": "กระติกน้ำ ThermoCup 500ml", "price": 7900}, {"text": "หม้อหุงข้าว RiceMate 1.8L ราคาปกติ 490 บาท", "product": "หม้อหุงข้าว RiceMate 1.8L", "price": 490}, {"text": "โปรพิเศษ เครื่องปั่น BlendGo เหลือ 2,990 บาท", "product": "เครื่องปั่น BlendGo", "price": 2990}, {"text": "โคมไฟอ่านหนังสือ ReadLite ราคาเพียง 1,890 บาท ส่งฟรี", "product": "โคมไฟอ่านหนังสือ ReadLite", "price": 1890}, {"text": "สั่ง พัดลมพกพา BreezePocket มาแล้ว 1,190 บาท คุ้มมาก", "product": "พัดลมพกพา BreezePocket", "price": 1190}, {"text": "รีวิว กระเป๋าเดินทาง RollEasy 24 นิ้ว ซื้อมา 990 บาท ใช้ดี", "product": "กระเป๋าเดินทาง RollEasy 24 นิ้ว", "price": 990}, {"text": "รีวิว รองเท้าผ้าใบ UrbanStep ซื้อมา 1,190 บาท ใช้ดี", "product": "รองเท้าผ้าใบ UrbanStep", "price": 1190}, {"text": "เสื้อแจ็กเก็ตกันลม WindGuard ราคา 1,290 บาท", "product": "เสื้อแจ็กเก็ตกันลม WindGuard", "price": 1290}, {"text": "นาฬิกาข้อมือ TimeCore ราคาปกติ 1,890 บาท", "product": "นาฬิกาข้อมือ TimeCore", "price": 1890}, {"text": "แว่นกันแดด SunVue ราคา 1,290 บาท", "product": "แว่นกันแดด SunVue", "price": 1290}, {"text": "หมวกแก๊ป CapOne ราคาปกติ 1,690 บาท", "product": "หมวกแก๊ป CapOne", "price": 1690}, {"text": "แท่นชาร์จไร้สาย ChargePad ลดเหลือ 3,990 บาท วันนี้เท่านั้น", "product": "แท่นชาร์จไร้สาย ChargePad", "price": 3990}, {"text": "สายชาร์จ Lightning FastLink ฿259 มีประกัน", "product": "สายชาร์จ Lightning FastLink", "price": 259}, {"text": "ที่วางโทรศัพท์ HoldUp ฿2,190 มีประกัน", "product": "ที่วางโทรศัพท์ HoldUp", "price": 2190}, {"text": "กล้องติดรถ DriveCam HD ฿7,900 มีประกัน", "product": "กล้องติดรถ DriveCam HD", "price": 7900}, {"text": "โปรพิเศษ เครื่องวัดความดัน HealthCheck เหลือ 4,590 บาท", "product": "เครื่องวัดความดัน HealthCheck", "price": 4590}, {"text": "ไม้เซลฟี่ SnapStick ราคา 2,490 บาท", "product": "ไม้เซลฟี่ SnapStick", "price": 2490}, {"text": "เต็นท์ 2 คน CampLite ราคา 3,990 บาท", "product": "เต็นท์ 2 คน CampLite", "price": 3990}, {"text": "ถุงนอน WarmNest ราคา 7,900 บาท", "product": "ถุงนอน WarmNest", "price": 7900}, {"text": "โปรพิเศษ ไฟฉาย LED BrightBeam เหลือ 1,490 บาท", "product": "ไฟฉาย LED BrightBeam", "price": 1490}, {"text": "เขียงไม้ ChopWood ตัวนี้ 5,900 บาท มีของแถม", "product": "เขียงไม้ ChopWood", "price": 5900}, {"text": "มีดเชฟ SharpEdge 8 นิ้ว ราคาปกติ 1,190 บาท", "product": "มีดเชฟ SharpEdge 8 นิ้ว", "price": 1190}, {"text": "ชุดเครื่องนอน SoftDream 6 ฟุต ฿4,590 มีประกัน", "product": "ชุดเครื่องนอน SoftDream 6 ฟุต", "price": 4590}, {"text": "หมอนยางพารา NeckEase ฿690 มีประกัน", "product": "หมอนยางพารา NeckEase", "price": 690}, {"text": "พรมเช็ดเท้า StepDry ฿690 มีประกัน", "product": "พรมเช็ดเท้า StepDry", "price": 690}, {"text": "ตะกร้าผ้า FoldBin ราคาปกติ 1,490 บาท", "product": "ตะกร้าผ้า FoldBin", "price": 1490}, {"text": "โปรพิเศษ หูฟังไร้สาย EchoBud เหลือ 990 บาท", "product": "หูฟังไร้สาย EchoBud", "price": 990}, {"text": "ขาย ลำโพงพกพา MiniBeat ในราคา 1,690 บาท", "product": "ลำโพงพกพา MiniBeat", "price": 1690}, {"text": "เมาส์เกมมิ่ง ClickX ราคาปกติ 199 บาท", "product": "เมาส์เกมมิ่ง ClickX", "price": 199}, {"text": "คีย์บอร์ดไร้สาย TypeLite ลดเหลือ 7,900 บาท วันนี้เท่านั้น", "product": "คีย์บอร์ดไร้สาย TypeLite", "price": 7900}, {"text": "รีวิว กระติกน้ำ ThermoCup 500ml ซื้อมา 1,190 บาท ใช้ดี", "product": "กระติกน้ำ ThermoCup 500ml", "price": 1190}, {"text": "โปรพิเศษ หม้อหุงข้าว RiceMate 1.8L เหลือ 890 บาท", "product": "หม้อหุงข้าว RiceMate 1.8L", "price": 890}, {"text": "เครื่องปั่น BlendGo ฿890 มีประกัน", "product": "เครื่องปั่น BlendGo", "price": 890}, {"text": "สั่ง โคมไฟอ่านหนังสือ ReadLite มาแล้ว 690 บาท คุ้มมาก", "product": "โคมไฟอ่านหนังสือ ReadLite", "price": 690}, {"text": "ขาย พัดลมพกพา BreezePocket ในราคา 5,900 บาท", "product": "พัดลมพกพา BreezePocket", "price": 5900}, {"text": "กระเป๋าเดินทาง RollEasy 24 นิ้ว ราคาปกติ 490 บาท", "product": "กระเป๋าเดินทาง RollEasy 24 นิ้ว", "price": 490}, {"text": "รองเท้าผ้าใบ UrbanStep ฿1,290 มีประกัน", "product": "รองเท้าผ้าใบ UrbanStep", "price": 1290}, {"text": "เสื้อแจ็กเก็ตกันลม WindGuard ราคาเพียง 7,900 บาท ส่งฟรี", "product": "เสื้อแจ็กเก็ตกันลม WindGuard", "price": 7900}, {"text": "นาฬิกาข้อมือ TimeCore ลดเหลือ 2,190 บาท วันนี้เท่านั้น", "product": "นาฬิกาข้อมือ TimeCore", "price": 2190}, {"text": "แว่นกันแดด SunVue ฿350 มีประกัน", "product": "แว่นกันแดด SunVue", "price": 350}, {"text": "หมวกแก๊ป CapOne ราคาเพียง 5,900 บาท ส่งฟรี", "product": "หมวกแก๊ป CapOne", "price": 5900}, {"text": "แท่นชาร์จไร้สาย ChargePad ตัวนี้ 3,990 บาท มีของแถม", "product": "แท่นชาร์จไร้สาย ChargePad", "price": 3990}, {"text": "สายชาร์จ Lightning FastLink ราคาเพียง 990 บาท ส่งฟรี", "product": "สายชาร์จ Lightning FastLink", "price": 990}, {"text": "ที่วางโทรศัพท์ HoldUp ราคา 490 บาท", "product": "ที่วางโทรศัพท์ HoldUp", "price": 490}, {"text": "กล้องติดรถ DriveCam HD ราคาเพียง 2,990 บาท ส่งฟรี", "product": "กล้องติดรถ DriveCam HD", "price": 2990}, {"text": "เครื่องวัดความดัน HealthCheck ราคาปกติ 1,290 บาท", "product": "เครื่องวัดความดัน HealthCheck", "price": 1290}, {"text": "โปรพิเศษ ไม้เซลฟี่ SnapStick เหลือ 690 บาท", "product": "ไม้เซลฟี่ SnapStick", "price": 690}, {"text": "ขาย เต็นท์ 2 คน CampLite ในราคา 1,290 บาท", "product": "เต็นท์ 2 คน CampLite", "price": 1290}, {"text": "ถุงนอน WarmNest ฿4,590 มีประกัน", "product": "ถุงนอน WarmNest", "price": 4590}, {"text": "ไฟฉาย LED BrightBeam ลดเหลือ 3,990 บาท วันนี้เท่านั้น", "product": "ไฟฉาย LED BrightBeam", "price": 3990}, {"text": "โปรพิเศษ เขียงไม้ ChopWood เหลือ 1,890 บาท", "product": "เขียงไม้ ChopWood", "price": 1890}, {"text": "โปรพิเศษ มีดเชฟ SharpEdge 8 นิ้ว เหลือ 2,990 บาท", "product": "มีดเชฟ SharpEdge 8 นิ้ว", "price": 2990}, {"text": "ชุดเครื่องนอน SoftDream 6 ฟุต ฿2,190 มีประกัน", "product": "ชุดเครื่องนอน SoftDream 6 ฟุต", "price": 2190}, {"text": "สั่ง หมอนยางพารา NeckEase มาแล้ว 5,900 บาท คุ้มมาก", "product": "หมอนยางพารา NeckEase", "price": 5900}, {"text": "สั่ง พรมเช็ดเท้า StepDry มาแล้ว 1,190 บาท คุ้มมาก", "product": "พรมเช็ดเท้า StepDry", "price": 1190}, {"text": "สั่ง ตะกร้าผ้า FoldBin มาแล้ว 3,990 บาท คุ้มมาก", "product": "ตะกร้าผ้า FoldBin", "price": 3990}]}, "classify": {"categories": ["การจัดส่ง", "สินค้าชำรุด", "การชำระเงิน", "บริการลูกค้า"], "dev": [{"text": "สั่งของมา 10 วันแล้วยังไม่ได้รับเลย สถานะขึ้นว่ากำลังจัดส่งตลอด", "label": "การจัดส่ง"}, {"text": "เปิดกล่องมาแล้วหน้าจอแตกเป็นรอยร้าว ใช้งานไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ตัดบัตรเครดิตไปแล้วสองครั้ง แต่ในระบบขึ้นว่ายังไม่ชำระ", "label": "การชำระเงิน"}, {"text": "โทรไปถามเรื่องสินค้า พนักงานพูดจาไม่ดีและวางสายใส่", "label": "บริการลูกค้า"}, {"text": "พัสดุมาถึงแต่กล่องบุบยับเยิน ของข้างในหักครึ่ง", "label": "สินค้าชำรุด"}, {"text": "แจ้งว่าจะส่งวันจันทร์ แต่เลื่อนมาเป็นศุกร์โดยไม่บอกล่วงหน้า", "label": "การจัดส่ง"}, {"text": "ขอคืนเงินไปสองสัปดาห์แล้ว ยังไม่ได้เงินคืนเข้าบัญชี", "label": "การชำระเงิน"}, {"text": "แชทถามไปตั้งแต่เช้า ตอบมาตอนสามทุ่มด้วยข้อความอัตโนมัติ", "label": "บริการลูกค้า"}, {"text": "ใช้โค้ดส่วนลดแล้วราคาไม่ลด แต่ระบบยังเรียกเก็บเต็มจำนวน", "label": "การชำระเงิน"}, {"text": "ได้ของผิดสี สั่งสีดำแต่ส่งสีขาวมา แถมมีรอยขีดข่วน", "label": "สินค้าชำรุด"}], "test": [{"text": "คนส่งของโยนกล่องไว้หน้าประตูแล้วไปโดยไม่กดกริ่ง ของหายไปเลย", "label": "การจัดส่ง"}, {"text": "หูฟังข้างซ้ายไม่มีเสียงตั้งแต่แกะกล่อง", "label": "สินค้าชำรุด"}, {"text": "ระบบหักเงินซ้ำสองรอบสำหรับออเดอร์เดียว", "label": "การชำระเงิน"}, {"text": "พนักงานหน้าร้านไม่ยอมรับเปลี่ยนสินค้าทั้งที่ยังอยู่ในระยะประกัน และพูดจาไม่สุภาพ", "label": "บริการลูกค้า"}, {"text": "เลขพัสดุที่ให้มาตรวจสอบไม่ได้ ขึ้นว่าไม่พบข้อมูล", "label": "การจัดส่ง"}, {"text": "เครื่องปั่นใช้ได้สองครั้งแล้วมอเตอร์ไหม้ มีกลิ่นพลาสติก", "label": "สินค้าชำรุด"}, {"text": "โอนเงินผ่านแอปธนาคารแล้ว แต่ออเดอร์ถูกยกเลิกอัตโนมัติว่าไม่ชำระ", "label": "การชำระเงิน"}, {"text": "ติดต่อคอลเซ็นเตอร์รอสายนาน 40 นาที แล้วสายหลุด", "label": "บริการลูกค้า"}, {"text": "สินค้ามาถึงช้ากว่ากำหนดหนึ่งสัปดาห์และไม่มีการแจ้งเตือนใด ๆ", "label": "การจัดส่ง"}, {"text": "โต๊ะที่ได้รับขาไม่เท่ากัน โยกไปมา ใช้งานไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ใบเสร็จระบุยอดไม่ตรงกับที่ถูกตัดบัตร ต่างกัน 300 บาท", "label": "การชำระเงิน"}, {"text": "แจ้งปัญหาไปสามครั้ง ได้แต่คำตอบว่าจะติดต่อกลับ แต่ไม่เคยติดต่อมา", "label": "บริการลูกค้า"}, {"text": "ส่งมาผิดที่อยู่ ทั้งที่กรอกที่อยู่ถูกต้องตอนสั่ง", "label": "การจัดส่ง"}, {"text": "รองเท้าที่ได้รับพื้นหลุดออกจากตัวรองเท้าตั้งแต่วันแรก", "label": "สินค้าชำรุด"}, {"text": "เลือกผ่อน 0% แต่ระบบเรียกเก็บเต็มจำนวนในงวดแรก", "label": "การชำระเงิน"}, {"text": "พนักงานแชทตอบไม่ตรงคำถาม ถามเรื่องประกันแต่ตอบเรื่องโปรโมชัน", "label": "บริการลูกค้า"}, {"text": "พัสดุถูกส่งกลับไปที่ผู้ขายโดยที่ไม่มีใครมาส่งเลยสักครั้ง", "label": "การจัดส่ง"}, {"text": "ซองบรรจุแตก เมล็ดกาแฟหกออกมาเกือบครึ่งถุง", "label": "สินค้าชำรุด"}, {"text": "ยกเลิกออเดอร์ตั้งแต่ก่อนส่ง แต่เงินยังไม่ถูกคืนหลังผ่านไป 20 วัน", "label": "การชำระเงิน"}, {"text": "ขอใบกำกับภาษีไปแล้วสองสัปดาห์ ยังไม่มีใครดำเนินการให้", "label": "บริการลูกค้า"}], "train": [{"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 10 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "กระเป๋ามีรอยขีดข่วนลึกหลายจุด เหมือนของมือสอง", "label": "สินค้าชำรุด"}, {"text": "แต้มสะสมถูกหักไปแล้ว แต่ส่วนลดไม่แสดงในยอดชำระ", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 10 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "ระบบให้เลือกวันส่ง แต่ของมาก่อนวันที่เลือกแล้วไม่มีคนรับ", "label": "การจัดส่ง"}, {"text": "กาต้มน้ำที่ส่งมาเป็นรุ่นเก่า ไม่ตรงกับที่โฆษณาไว้", "label": "สินค้าชำรุด"}, {"text": "เปลี่ยนวิธีจ่ายเป็นเก็บเงินปลายทาง แต่ระบบยังตัดบัตรอยู่ดี", "label": "การชำระเงิน"}, {"text": "ขอเอกสารรับประกันไปแล้ว 3 ครั้ง ยังไม่ได้รับ", "label": "บริการลูกค้า"}, {"text": "เปลี่ยนที่อยู่จัดส่งในระบบแล้ว แต่ยังส่งไปที่อยู่เก่า", "label": "การจัดส่ง"}, {"text": "เก้าอี้ใช้ไปแค่ 3 วัน ปุ่มก็กดไม่ติดแล้ว", "label": "สินค้าชำรุด"}, {"text": "ราคาที่หน้าตะกร้ากับหน้าชำระเงินไม่ตรงกัน ต่างกัน 300 บาท", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 7 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "กล่องใหญ่ถูกยัดใส่ตู้รับพัสดุจนต้องดึงออกไม่ได้", "label": "การจัดส่ง"}, {"text": "ขนาดของหูฟังเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "ราคาที่หน้าตะกร้ากับหน้าชำระเงินไม่ตรงกัน ต่างกัน 200 บาท", "label": "การชำระเงิน"}, {"text": "ช่องทางติดต่อบนเว็บเป็นเบอร์ที่โทรไม่ติดทุกครั้ง", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 7 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "ขนาดของลำโพงเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "สั่งซื้อด้วยคูปองเงินสด แต่ยอดคูปองหายไปและไม่ได้สินค้า", "label": "การชำระเงิน"}, {"text": "แชทบอทวนตอบเรื่องเดิม ไม่มีทางต่อสายกับคนจริง", "label": "บริการลูกค้า"}, {"text": "แอปแจ้งว่าส่งแล้ว แต่รูปหลักฐานเป็นหน้าบ้านคนอื่น", "label": "การจัดส่ง"}, {"text": "สีของพัดลมหลุดลอกหลังจากล้างครั้งเดียว", "label": "สินค้าชำรุด"}, {"text": "โอนเงินเกินไป 200 บาทเพราะระบบแสดงยอดผิด ยังไม่ได้เงินส่วนเกินคืน", "label": "การชำระเงิน"}, {"text": "พนักงานให้ข้อมูลโปรโมชันผิด ทำให้เสียสิทธิ์ส่วนลด", "label": "บริการลูกค้า"}, {"text": "แจ้งเตือนว่าจะส่งวันศุกร์ รอทั้งวันไม่มีใครมา ไม่มีข้อความอะไรเลย", "label": "การจัดส่ง"}, {"text": "ซีลของสายชาร์จถูกแกะแล้ว และมีคราบสกปรกข้างใน", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่านวอลเล็ตแล้ว 5 วันยังไม่ได้รับใบเสร็จอิเล็กทรอนิกส์", "label": "การชำระเงิน"}, {"text": "ถามเรื่องสเปกสินค้า พนักงานบอกให้ไปหาเองในอินเทอร์เน็ต", "label": "บริการลูกค้า"}, {"text": "รอบส่งเลื่อนมาแล้ว 2 ครั้งภายในสัปดาห์เดียว", "label": "การจัดส่ง"}, {"text": "ขนาดของกระจกเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "ยอดที่ตัดบัตรมากกว่าราคาที่แสดงตอนกดสั่ง 200 บาท", "label": "การชำระเงิน"}, {"text": "ขอคุยกับหัวหน้าแล้วถูกบอกว่าไม่มีใครรับผิดชอบเรื่องนี้", "label": "บริการลูกค้า"}, {"text": "จ่ายค่าส่งด่วนแล้ว แต่ของมาถึงช้ากว่าแบบธรรมดา", "label": "การจัดส่ง"}, {"text": "ซีลของลำโพงถูกแกะแล้ว และมีคราบสกปรกข้างใน", "label": "สินค้าชำรุด"}, {"text": "เงินคืนจากการยกเลิกออเดอร์ได้มาไม่ครบ หายไป 500 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานหน้าร้านมองข้ามลูกค้าที่ยืนรอ แล้วไปคุยโทรศัพท์ส่วนตัว", "label": "บริการลูกค้า"}, {"text": "ของสดถูกวางตากแดดไว้หน้าบ้านทั้งวันกว่าจะมีคนบอก", "label": "การจัดส่ง"}, {"text": "กระเป๋าทำงานได้แค่ครึ่งเดียว อีกครึ่งไม่ตอบสนอง", "label": "สินค้าชำรุด"}, {"text": "ระบบเก็บค่าธรรมเนียมเพิ่มโดยไม่แจ้งก่อนกดชำระ", "label": "การชำระเงิน"}, {"text": "พนักงานปิดเคสให้เองทั้งที่ปัญหายังไม่ถูกแก้", "label": "บริการลูกค้า"}, {"text": "แจ้งเตือนว่าจะส่งวันพุธ รอทั้งวันไม่มีใครมา ไม่มีข้อความอะไรเลย", "label": "การจัดส่ง"}, {"text": "ขนาดของสายชาร์จเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "สมัครสมาชิกรายเดือนแล้ว แต่ถูกเรียกเก็บเป็นรายปี", "label": "การชำระเงิน"}, {"text": "ร้องเรียนเรื่องพนักงานแล้วถูกบล็อกจากเพจของร้าน", "label": "บริการลูกค้า"}, {"text": "แยกส่งเป็นสองกล่อง อีกกล่องหายไปเฉย ๆ ไม่มีใครตามให้", "label": "การจัดส่ง"}, {"text": "กาต้มน้ำมีกลิ่นเหม็นสารเคมีแรงมาก ใช้ไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ระบบแจ้งว่าบัตรถูกปฏิเสธ แต่ธนาคารยืนยันว่าตัดเงินไปแล้ว", "label": "การชำระเงิน"}, {"text": "พนักงานตอบแชทด้วยอีโมจิอย่างเดียว ไม่ตอบคำถามที่ถาม", "label": "บริการลูกค้า"}, {"text": "ขนส่งโทรมาแล้ววางสายทันที ไม่รอให้รับ แล้วตีกลับว่าติดต่อไม่ได้", "label": "การจัดส่ง"}, {"text": "เปิดกล่องมาลำโพงเปียกชื้น มีคราบน้ำอยู่ข้างใน", "label": "สินค้าชำรุด"}, {"text": "ยอดที่ตัดบัตรมากกว่าราคาที่แสดงตอนกดสั่ง 500 บาท", "label": "การชำระเงิน"}, {"text": "อีเมลแจ้งปัญหาไปแล้ว 3 ฉบับ ได้รับแต่ตอบรับอัตโนมัติ", "label": "บริการลูกค้า"}, {"text": "รอบส่งเลื่อนมาแล้ว 4 ครั้งภายในสัปดาห์เดียว", "label": "การจัดส่ง"}, {"text": "กาต้มน้ำมีรอยแตกร้าวมาตั้งแต่ในกล่อง ดูเหมือนโดนกระแทกระหว่างทาง", "label": "สินค้าชำรุด"}, {"text": "ผ่อนชำระเดือนแรกถูกตัดสองงวดพร้อมกัน", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 3 วัน", "label": "บริการลูกค้า"}, {"text": "ขนส่งบอกว่าที่อยู่ไม่ชัดเจน ทั้งที่เคยส่งที่เดิมมาแล้วหลายครั้ง", "label": "การจัดส่ง"}, {"text": "ซีลของตัวเครื่องถูกแกะแล้ว และมีคราบสกปรกข้างใน", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่านวอลเล็ตแล้ว 30 วันยังไม่ได้รับใบเสร็จอิเล็กทรอนิกส์", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 10 วัน", "label": "บริการลูกค้า"}, {"text": "กรอกที่อยู่คอนโดพร้อมเลขห้อง แต่ของถูกฝากไว้ที่ร้านสะดวกซื้ออีกซอย", "label": "การจัดส่ง"}, {"text": "หลังชาร์จหน้าจอครั้งแรก ตัวเครื่องร้อนจนจับไม่ได้", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่านวอลเล็ตแล้ว 3 วันยังไม่ได้รับใบเสร็จอิเล็กทรอนิกส์", "label": "การชำระเงิน"}, {"text": "อีเมลแจ้งปัญหาไปแล้ว 2 ฉบับ ได้รับแต่ตอบรับอัตโนมัติ", "label": "บริการลูกค้า"}, {"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 5 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "หลังชาร์จสายชาร์จครั้งแรก ตัวเครื่องร้อนจนจับไม่ได้", "label": "สินค้าชำรุด"}, {"text": "โปรผ่อน 0% หายไปตอนกดชำระ กลายเป็นคิดดอกเบี้ยเต็ม", "label": "การชำระเงิน"}, {"text": "นัดช่างมาซ่อมที่บ้านแล้วไม่มา และไม่แจ้งยกเลิก", "label": "บริการลูกค้า"}, {"text": "ไรเดอร์เอาของไปส่งผิดบ้าน ต้องเดินไปขอคืนจากเพื่อนบ้านเอง", "label": "การจัดส่ง"}, {"text": "ลำโพงมีรอยแตกร้าวมาตั้งแต่ในกล่อง ดูเหมือนโดนกระแทกระหว่างทาง", "label": "สินค้าชำรุด"}, {"text": "เงินมัดจำถูกหักแล้ว แต่ออเดอร์ไม่ถูกสร้างในระบบ", "label": "การชำระเงิน"}, {"text": "ถูกโอนสายวนไป 2 แผนก สุดท้ายกลับมาที่เดิม", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 14 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "ขาโต๊ะมีกลิ่นเหม็นสารเคมีแรงมาก ใช้ไม่ได้", "label": "สินค้าชำรุด"}, {"text": "เงินคืนจากการยกเลิกออเดอร์ได้มาไม่ครบ หายไป 300 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 7 วัน", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 10 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "หลังชาร์จกระจกครั้งแรก ตัวเครื่องร้อนจนจับไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ขอเปลี่ยนใบเสร็จเป็นชื่อบริษัทแล้วถูกเรียกเก็บเงินเพิ่ม", "label": "การชำระเงิน"}, {"text": "พนักงานพูดจาประชดเมื่อถามเรื่องการคืนสินค้า", "label": "บริการลูกค้า"}, {"text": "พนักงานส่งเรียกเก็บเงินปลายทางซ้ำ ทั้งที่จ่ายออนไลน์แล้ว", "label": "การจัดส่ง"}, {"text": "ลำโพงมีเสียงดังผิดปกติตอนเปิดใช้ครั้งแรก", "label": "สินค้าชำรุด"}, {"text": "ยอดที่ตัดบัตรมากกว่าราคาที่แสดงตอนกดสั่ง 100 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 20 วัน", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 5 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "ขาโต๊ะมีรอยขีดข่วนลึกหลายจุด เหมือนของมือสอง", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่าน QR แล้วเงินออกจากบัญชี แต่หน้าเว็บขึ้นว่าชำระไม่สำเร็จ", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 30 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 3 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "หูฟังมีกลิ่นเหม็นสารเคมีแรงมาก ใช้ไม่ได้", "label": "สินค้าชำรุด"}, {"text": "ราคาที่หน้าตะกร้ากับหน้าชำระเงินไม่ตรงกัน ต่างกัน 500 บาท", "label": "การชำระเงิน"}, {"text": "อีเมลแจ้งปัญหาไปแล้ว 4 ฉบับ ได้รับแต่ตอบรับอัตโนมัติ", "label": "บริการลูกค้า"}, {"text": "รอบส่งเลื่อนมาแล้ว 3 ครั้งภายในสัปดาห์เดียว", "label": "การจัดส่ง"}, {"text": "กระจกขึ้นสนิมภายในสัปดาห์แรก", "label": "สินค้าชำรุด"}, {"text": "เงินคืนจากการยกเลิกออเดอร์ได้มาไม่ครบ หายไป 200 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 30 วัน", "label": "บริการลูกค้า"}, {"text": "แจ้งเตือนว่าจะส่งวันจันทร์ รอทั้งวันไม่มีใครมา ไม่มีข้อความอะไรเลย", "label": "การจัดส่ง"}, {"text": "ขนาดของพัดลมเล็กกว่าที่ระบุในหน้าสินค้าเกือบ 5 ซม.", "label": "สินค้าชำรุด"}, {"text": "จ่ายผ่านวอลเล็ตแล้ว 20 วันยังไม่ได้รับใบเสร็จอิเล็กทรอนิกส์", "label": "การชำระเงิน"}, {"text": "ถูกโอนสายวนไป 4 แผนก สุดท้ายกลับมาที่เดิม", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 3 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "เก้าอี้ประกอบไม่ได้เพราะรูสกรูไม่ตรงกัน", "label": "สินค้าชำรุด"}, {"text": "เงินคืนจากการยกเลิกออเดอร์ได้มาไม่ครบ หายไป 100 บาท", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 5 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 30 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "ลำโพงมีรอยขีดข่วนลึกหลายจุด เหมือนของมือสอง", "label": "สินค้าชำรุด"}, {"text": "โอนเงินเกินไป 300 บาทเพราะระบบแสดงยอดผิด ยังไม่ได้เงินส่วนเกินคืน", "label": "การชำระเงิน"}, {"text": "ถูกโอนสายวนไป 3 แผนก สุดท้ายกลับมาที่เดิม", "label": "บริการลูกค้า"}, {"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 20 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "พัดลมทำงานได้แค่ครึ่งเดียว อีกครึ่งไม่ตอบสนอง", "label": "สินค้าชำรุด"}, {"text": "โอนเงินเกินไป 100 บาทเพราะระบบแสดงยอดผิด ยังไม่ได้เงินส่วนเกินคืน", "label": "การชำระเงิน"}, {"text": "แจ้งเคลมประกันไป 3 วัน ไม่มีใครติดต่อกลับเลย", "label": "บริการลูกค้า"}, {"text": "ส่งไปต่างจังหวัด 20 วันยังไม่ถึง ทั้งที่แจ้งว่า 2 วัน", "label": "การจัดส่ง"}, {"text": "สายชาร์จมีเสียงดังผิดปกติตอนเปิดใช้ครั้งแรก", "label": "สินค้าชำรุด"}, {"text": "ยอดที่ตัดบัตรมากกว่าราคาที่แสดงตอนกดสั่ง 300 บาท", "label": "การชำระเงิน"}, {"text": "ขอเอกสารรับประกันไปแล้ว 2 ครั้ง ยังไม่ได้รับ", "label": "บริการลูกค้า"}, {"text": "พัสดุค้างอยู่ที่ศูนย์คัดแยก 30 วันแล้ว สถานะไม่ขยับเลย", "label": "การจัดส่ง"}, {"text": "เก้าอี้ที่ส่งมาเป็นรุ่นเก่า ไม่ตรงกับที่โฆษณาไว้", "label": "สินค้าชำรุด"}, {"text": "ราคาที่หน้าตะกร้ากับหน้าชำระเงินไม่ตรงกัน ต่างกัน 100 บาท", "label": "การชำระเงิน"}, {"text": "พนักงานสัญญาว่าจะส่งอะไหล่ให้ฟรี แล้วเงียบหายไป 5 วัน", "label": "บริการลูกค้า"}]}}''')
for task in ("extract", "classify"):
    print(task, {k: len(v) for k, v in DATA[task].items() if isinstance(v, list)})
print("ตัวอย่าง extract/dev:", DATA["extract"]["dev"][0])
print("ตัวอย่าง classify/dev:", DATA["classify"]["dev"][0])

In [ ]:
#@title เกณฑ์ให้คะแนน — ชุดเดียวกับ Lab 3 ทุกตัวอักษร (โค้ดถูกซ่อน ดับเบิลคลิกถ้าอยากดู) { display-mode: "form" }
def strip_wrap(s):
    # ตัดคำนำหน้า "คำตอบ:" และ ``` ที่ห่อ JSON ออก (ยอมให้มีได้ ไม่นับว่าผิดรูปแบบ)
    s = re.sub(r"^(คำตอบ|answer)\s*[:：]\s*", "", s.strip(), flags=re.I)
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", s).strip()

def parse_json_obj(s):
    # ใช้ให้คะแนน "เนื้อหา": ลอง parse ทั้งคำตอบก่อน ถ้ามีข้อความปนค่อยหยิบก้อน {...} ก้อนแรก
    try:
        o = json.loads(strip_wrap(s))
        if isinstance(o, dict): return o
    except Exception: pass
    m = re.search(r"\{.*?\}", s, re.S)
    if not m: return None
    try:
        o = json.loads(m.group(0)); return o if isinstance(o, dict) else None
    except Exception: return None

def norm(s): return re.sub(r"\s+", " ", str(s)).strip()
def score_extract(output, item):
    # product ตรงเป๊ะ 0.5 · ตรงบางส่วน (ชื่อสั้น/ยาวกว่าเฉลย) 0.25 · price ตรง 0.5
    obj = parse_json_obj(output)
    if obj is None: return 0.0
    s = 0.0
    p_out, p_gold = norm(obj.get("product", "")), norm(item["product"])
    if p_out == p_gold: s += 0.5
    elif p_out and (p_out in p_gold or p_gold in p_out): s += 0.25
    try:
        if int(float(str(obj.get("price", "")).replace(",", "").replace("฿", ""))) == item["price"]: s += 0.5
    except Exception: pass
    return s

CATS = DATA["classify"]["categories"]
def first_answer(output):
    # บรรทัดแรกของคำตอบ ตัดคำนำหน้า "หมวด:" / "คำตอบ:" และเครื่องหมายหัวท้ายออก
    line = output.strip().split("\n")[0]
    line = re.sub(r"^(หมวด|คำตอบ)\s*[:：]\s*", "", line.strip())
    return line.strip(" .!\"'`*")
def score_classify(output, item):
    return 1.0 if first_answer(output) == item["label"] else 0.0     # บรรทัดแรกต้องเป็นชื่อหมวดที่ถูก "พอดี"

def format_ok(task, output):
    # "ตอบตรงรูปแบบ" = ระบบปลายทางเอาไปใช้ได้ทันทีโดยไม่ต้องตัดต่อ
    if task == "classify": return first_answer(output) in CATS      # บรรทัดแรกเป็นชื่อหมวดใดหมวดหนึ่ง
    try: obj = json.loads(strip_wrap(output))                         # ทั้งคำตอบต้องเป็น JSON (ห้ามมีคำอธิบายปน)
    except Exception: return False
    return isinstance(obj, dict) and "product" in obj and "price" in obj

## 4.1 เลือกโจทย์ แล้วแปลงข้อมูลเป็นบทสนทนา
ข้อมูลเทรน 1 ข้อ = บทสนทนา 1 ก้อน: **system สั้น + ข้อความ (user) + คำตอบ (assistant)**

เทียบกับโมเดลภาพ: input = ภาพ · label = ชื่อคลาส — ที่นี่
- **input** = ทุก token ในก้อน (โมเดลเห็นทั้งหมด)
- **label** = ก้อนเดียวกัน แต่ตำแหน่งที่เป็น system / ข้อความลูกค้า ตั้งเป็น **−100 = ไม่คิด loss** → เหลือคิด loss เฉพาะคำตอบ + token จบ `<|eot_id|>`
  (โมเดลยังเห็นส่วนที่เป็น −100 และใช้ attention ดึงข้อมูลจากมันได้ แค่ไม่ถูก "ให้คะแนน" ตรงนั้น · โมเดลเลื่อนให้เอง 1 ตำแหน่ง: token ที่ i ใช้ทาย token ที่ i+1)

ทำไมไม่คิด loss ที่ส่วนคำถาม: เราอยากสอน "วิธีตอบ" ไม่ได้อยากให้โมเดลเก่งเดาว่าลูกค้าจะบ่นอะไร · **system prompt สั้นตัวนี้ต้องใช้ตอนใช้งานด้วย**

In [ ]:
TASK = "classify"        # เลือก "extract" หรือ "classify" ตามผล Lab 3 (โจทย์ที่ prompt เต็มแล้วยังไม่ถึงเป้า)

# system สั้นมาก — รายชื่อหมวด/รูปแบบ JSON ไม่ต้องบอก เพราะโมเดลจะเรียนจากตัวอย่าง 100 ข้อเอง
if TASK == "extract":
    SYSTEM = "สกัดชื่อสินค้าและราคาเป็น JSON"
    def answer_of(it): return json.dumps({"product": it["product"], "price": it["price"]}, ensure_ascii=False)
else:
    SYSTEM = "คัดแยกคำร้องเรียนเป็นหมวด"
    def answer_of(it): return it["label"]

def encode_prompt(system, user):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)   # จบด้วยหัว assistant
    return tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE)

def build_example(it):
    ids_prompt = encode_prompt(SYSTEM, it["text"])[0].tolist()                        # system + ข้อความ + หัว assistant
    ids_answer = tok(answer_of(it), add_special_tokens=False).input_ids + [EOT]        # คำตอบ + token จบ
    return {"input_ids": ids_prompt + ids_answer,
            "attention_mask": [1] * (len(ids_prompt) + len(ids_answer)),
            "labels": [-100] * len(ids_prompt) + ids_answer}                          # −100 = ไม่คิด loss

items = list(DATA[TASK]["train"]); random.Random(0).shuffle(items)      # copy ก่อนสับ → รันซ้ำได้ผลเดิม
train_ex = [build_example(x) for x in items[:100]]
eval_ex  = [build_example(x) for x in items[100:]]                     # eval = ดูระหว่างเทรน (ไม่ใช่ชุด test)
print(f"train {len(train_ex)} ข้อ | eval {len(eval_ex)} ข้อ")

e = train_ex[0]
print("\n① ทั้งก้อนที่โมเดลเห็น (input):\n" + tok.decode(e["input_ids"]))
print("\n② ส่วนที่คิด loss (label ที่ไม่ใช่ −100):", repr(tok.decode([t for t, l in zip(e["input_ids"], e["labels"]) if l != -100])))
start = next(i for i, l in enumerate(e["labels"]) if l != -100)          # ตำแหน่งแรกของคำตอบ
print(f"\n③ ดูทีละ token ตรงรอยต่อ (ทั้งก้อน {len(e['input_ids'])} token · {start} token แรกเป็น −100):")
for t, l in list(zip(e["input_ids"], e["labels"]))[start - 4:]:
    print(f"   {tok.decode([t])!r:32s} {'คิด loss ✓' if l != -100 else '−100 (ไม่คิด loss)'}")
print("   (ถ้าเห็น '�' = เศษของอักษรไทยที่ถูกตัดกลางตัว — ต่อกันแล้วจะเป็นคำปกติ)")

## 4.2 ใส่ LoRA แล้วนับพารามิเตอร์ที่เทรนจริง
**คำนวณก่อนรัน:** ต่อชั้น Q: 2,048×16 + 16×2,048 · K: 2,048×16 + 16×512 · V เหมือน K · O เหมือน Q → รวม 16 ชั้น ≈ ? ล้าน

In [ ]:
from peft import LoraConfig, get_peft_model
if hasattr(model, "peft_config"):
    raise RuntimeError("โมเดลใส่ LoRA ไปแล้ว — ถ้าจะเริ่มใหม่ ให้รันเซลล์โหลดโมเดลอีกครั้ง")
set_seed(0)                                                # ค่าเริ่มต้นของตาราง LoRA เหมือนกันทุกครั้งที่รัน
r = 16                                                     # rank ของ LoRA
model = get_peft_model(model, LoraConfig(r=r, lora_alpha=16, lora_dropout=0.0, bias="none",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],   # attention 4 ตาราง
                                         task_type="CAUSAL_LM"))
for p in model.parameters():
    if p.requires_grad: p.data = p.data.float()            # ตาราง LoRA เก็บเป็น fp32 → เทรนแบบ fp16 ได้เสถียร
trainable = sum(p.numel() for n_, p in model.named_parameters() if "lora_" in n_)
total     = sum(p.numel() for p in model.parameters())
print(f"โค้ดนับได้: เทรน {trainable/1e6:.2f} ล้าน จากทั้งหมด {total/1e6:,.1f} ล้าน = {trainable/total*100:.2f}%")

H, L = model.config.hidden_size, model.config.num_hidden_layers
kv = model.config.num_key_value_heads * (H // model.config.num_attention_heads)          # 8 × 64 = 512 (GQA)
per_layer = (H*r + r*H) + (H*r + r*kv) + (H*r + r*kv) + (H*r + r*H)                       # Q, K, V, O
print(f"คำนวณมือ:  {per_layer:,} ต่อชั้น × {L} ชั้น = {per_layer*L/1e6:.2f} ล้าน")

## 4.3 เทรน
3 epoch × 100 ข้อ / batch 8 ≈ 39 step · วัด eval loss ทุก 5 step · ตั้ง fp16 เพราะ T4 ไม่รองรับ bf16
ระหว่างเทรนจะพิมพ์ train loss / eval loss ทุก 5 step (บรรทัดละครั้ง) แล้ววาดกราฟตอนจบ

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, TrainerCallback
from transformers.trainer_callback import PrinterCallback
if "trainer" in globals():
    raise RuntimeError("เทรนไปแล้ว — รันซ้ำจะเทรนต่อจากของเดิม ถ้าจะเริ่มใหม่ ให้รันตั้งแต่เซลล์โหลดโมเดล")
tok.padding_side = "right"                                   # เติม padding ด้านขวาตอนเทรน
args = TrainingArguments(
    output_dir="lab4_out", num_train_epochs=3, per_device_train_batch_size=8,
    learning_rate=2e-4, warmup_steps=5, lr_scheduler_type="linear", weight_decay=0.01,
    logging_steps=5, eval_strategy="steps", eval_steps=5, save_strategy="no",
    fp16=torch.cuda.is_available(), label_names=["labels"], report_to="none", seed=0, disable_tqdm=True)

class ShowLoss(TrainerCallback):                             # พิมพ์ 1 บรรทัดทุกครั้งที่วัด eval (แทน log ยาว ๆ ของ Trainer)
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        tr = [x["loss"] for x in state.log_history if "loss" in x]
        print(f"step {state.global_step:2d}/{state.max_steps} · train loss {tr[-1] if tr else float('nan'):.3f} · eval loss {metrics['eval_loss']:.3f}")

trainer = Trainer(model=model, args=args, callbacks=[ShowLoss()],
                  train_dataset=Dataset.from_list(train_ex), eval_dataset=Dataset.from_list(eval_ex),
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))   # padding ของ label = −100
trainer.remove_callback(PrinterCallback)
trainer.train()
trainer.evaluate()                                           # วัด eval ครั้งสุดท้ายหลังจบ (step 39)
model.eval()

logs = trainer.state.log_history
tr = [(x["epoch"], x["loss"]) for x in logs if "loss" in x]
ev = [(x["epoch"], x["eval_loss"]) for x in logs if "eval_loss" in x]
plt.figure(figsize=(7, 3.6)); plt.plot(*zip(*tr), label="train loss"); plt.plot(*zip(*ev), "o-", label="eval loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("ถ้า eval loss ขึ้นขณะ train loss ยังลง = เริ่มท่องจำ (overfitting)"); plt.show()
print(f"loss เริ่ม {tr[0][1]:.2f} → จบ train {tr[-1][1]:.3f} · eval {ev[-1][1]:.3f}")

**ควรเห็นอะไร:** loss ลดลงเร็วตั้งแต่ epoch แรก แล้วค่อย ๆ ลู่เข้าใกล้ 0 (คำตอบสั้นและมีรูปแบบตายตัว เรียนได้เร็ว)

ชุด eval นี้ตัดมาจากข้อมูลที่สร้างจากแม่แบบชุดเดียวกับ train จึง **ง่ายพอ ๆ กับ train** — มักเห็นสองเส้นลงไปด้วยกัน ซึ่ง **ไม่ได้แปลว่าโมเดลเก่งจริง**
ตัวตัดสินจริงคือชุด test ที่เขียนด้วยมือใน 4.4 · ถ้า eval loss ขึ้นขณะ train loss ยังลง (แบบในสไลด์) = เริ่มท่องจำ → ลด epoch หรือเพิ่มข้อมูลที่หลากหลายขึ้น

## 4.4 เปิดชุด test ครั้งแรก — เทียบ 3 แนวทาง
1. โมเดลเดิม + zero-shot (L0)
2. โมเดลเดิม + prompt เต็มที่ดีที่สุดจาก Lab 3 (จาก `best_prompts.json` ถ้าอัปโหลดมา · ไม่มีก็ใช้ L3 ที่ฝังไว้)
3. โมเดล fine-tune + prompt สั้น (`SYSTEM` เดียวกับตอนเทรน + ข้อความล้วน)

ทั้งสามใช้โมเดลหลักตัวเดียวกันในหน่วยความจำ — แนวทาง (1)(2) แค่ **ปิด LoRA ชั่วคราว** (`disable_adapter`)

**คอลัมน์ในตาราง:** คะแนนเฉลี่ย (0–1 เกณฑ์เดียวกับ "คะแนน dev" ใน Lab 3) · ถูกทั้งข้อ (ได้ 1 เต็ม) · ตอบตรงรูปแบบ · token ของ prompt (= ต้นทุนต่อครั้ง)

**อ่านผลอย่างไร:** 20 ข้อยังน้อย ความต่างเล็ก ๆ อาจเป็นความบังเอิญ — โค้ดคำนวณค่า p ให้ (McNemar: ดูเฉพาะข้อที่สองแนวทางตอบต่างกัน ถ้าฝั่งหนึ่งชนะเกือบทุกข้อที่ต่างกัน p จะเล็ก)
**p < 0.05 = "ต่างจริง"** · p 0.05–0.20 = "มีแนวโน้ม" · p มากกว่านั้น = "ยังสรุปไม่ได้"

In [ ]:
@torch.no_grad()
def ask(system, user, adapter=True, max_new_tokens=80):
    ids = encode_prompt(system, user)
    ctx = contextlib.nullcontext() if adapter else model.disable_adapter()     # adapter=False → โมเดลเดิม
    with ctx:
        g = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens,
                           do_sample=False, eos_token_id=EOT, pad_token_id=EOT)
    return tok.decode(g[0, ids.shape[1]:], skip_special_tokens=True).strip(), ids.shape[1]
scorer = score_extract if TASK == "extract" else score_classify

# ---- prompt ของแต่ละแนวทาง (L0 และ L3 เหมือน Lab 3 ทุกตัวอักษร) ----
if TASK == "extract":
    L0 = (None, lambda x: f"สกัดชื่อสินค้าและราคาจากข้อความนี้: {x}")
    L3 = ("คุณคือระบบสกัดข้อมูลสินค้าจากประกาศขายภาษาไทย",
          lambda x: ('สกัดชื่อสินค้าและราคาจากข้อความ ตอบเป็น JSON เท่านั้น รูปแบบ {"product": ..., "price": <ตัวเลขไม่มีเครื่องหมาย>}\n'
                     'ข้อความ: นาฬิกาปลุก WakeUp Mini ราคา 350 บาท\nคำตอบ: {"product": "นาฬิกาปลุก WakeUp Mini", "price": 350}\n'
                     'ข้อความ: ขายพรมห้องนั่งเล่น CozyRug 2x3 เมตร ในราคา 2,190 บาท ส่งฟรี\nคำตอบ: {"product": "พรมห้องนั่งเล่น CozyRug 2x3 เมตร", "price": 2190}\n'
                     f'ข้อความ: {x}\nคำตอบ:'))
else:
    CAT_STR = " / ".join(CATS)
    L0 = (None, lambda x: f"คำร้องเรียนนี้อยู่ในหมวดไหน: {x}")
    L3 = ("คุณคือเจ้าหน้าที่คัดแยกคำร้องเรียนของร้านค้าออนไลน์",
          lambda x: (f"จัดหมวดหมู่คำร้องเรียนต่อไปนี้ เลือกหนึ่งหมวดจาก [{CAT_STR}] ตอบเฉพาะชื่อหมวด\nคำร้องเรียน: ของมาถึงช้ากว่ากำหนดสามวัน\nหมวด: การจัดส่ง\n"
                     f"คำร้องเรียน: แก้วที่ได้รับมีรอยร้าว\nหมวด: สินค้าชำรุด\nคำร้องเรียน: {x}\nหมวด:"))
BEST_NAME = "L3 + บทบาท (ฝังไว้)"
files = sorted(glob.glob("best_prompts*.json"))          # รองรับชื่อ "best_prompts (1).json" ที่ได้จากการดาวน์โหลดซ้ำ
try:
    bp = json.load(open(files[-1], encoding="utf-8"))[TASK]
    if bp["level"].startswith("L0"):
        print("best_prompts.json เป็น L0 → แนวทาง (2) ต้องเป็น prompt เต็ม จึงใช้ L3 ที่ฝังไว้แทน")
    else:
        L3 = (bp["system"], lambda x, t=bp["user_template"]: t.replace("{TEXT}", x)); BEST_NAME = bp["level"] + " (จาก Lab 3)"
        print(f"ใช้ prompt ที่ดีที่สุดจาก {files[-1]}:", bp["level"])
except (IndexError, KeyError, json.JSONDecodeError):
    print("ไม่พบ best_prompts.json (หรืออ่านไม่ได้) → ใช้ L3 ที่ฝังไว้")

ARMS = {"(1) โมเดลเดิม + zero-shot":        (*L0, False),
        f"(2) โมเดลเดิม + {BEST_NAME}":      (*L3, False),
        "(3) fine-tune + prompt สั้น":       (SYSTEM, lambda x: x, True)}
arm1, arm2, arm3 = ARMS                                   # ชื่อของแต่ละแนวทาง
final, rows = {}, {}
for name, (system, fn, adapter) in ARMS.items():
    rows[name] = []
    for it in DATA[TASK]["test"]:
        out, n = ask(system, fn(it["text"]), adapter)
        rows[name].append({"text": it["text"], "output": out, "score": scorer(out, it), "fmt": format_ok(TASK, out), "tokens": n})
    R = rows[name]
    final[name] = {"คะแนนเฉลี่ย (0–1)": round(sum(x["score"] for x in R) / len(R), 2),
                   "ถูกทั้งข้อ": f'{sum(x["score"] == 1 for x in R)}/{len(R)}',
                   "ตอบตรงรูปแบบ": f'{sum(x["fmt"] for x in R)}/{len(R)}',
                   "token ของ prompt (เฉลี่ย)": round(sum(x["tokens"] for x in R) / len(R))}
    print("เสร็จ:", name)
display(pd.DataFrame.from_dict(final, orient="index"))

def compare(a, b):
    only_a = sum(x["score"] == 1 and y["score"] < 1 for x, y in zip(rows[a], rows[b]))   # a ถูกทั้งข้อ แต่ b ไม่
    only_b = sum(y["score"] == 1 and x["score"] < 1 for x, y in zip(rows[a], rows[b]))   # b ถูกทั้งข้อ แต่ a ไม่
    n = only_a + only_b
    p = 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(min(only_a, only_b) + 1)) / 2**n)
    verdict = "ต่างจริง" if p < 0.05 else ("มีแนวโน้ม" if p < 0.20 else "ยังสรุปไม่ได้")
    A, B = a[:3], b[:3]
    print(f"{B} เทียบ {A}: ถูกทั้งข้อต่าง {only_b - only_a:+d} ข้อ · {B} ถูกแต่ {A} ผิด = {only_b} · {A} ถูกแต่ {B} ผิด = {only_a} · p = {p:.3f} → {verdict}")
compare(arm2, arm3); compare(arm1, arm3)

## ✍️ สรุปของคุณ
1. fine-tune ชนะ prompt เต็มไหมสำหรับโจทย์นี้? ใช้คำตามที่โค้ดพิมพ์ (ต่างจริง / มีแนวโน้ม / ยังสรุปไม่ได้)
2. ถ้าชนะ ชนะที่คะแนน หรือชนะที่ prompt สั้นลง (ต้นทุนต่อครั้งต่ำลง)? ดูคอลัมน์ token — ถ้าแนวทาง (2) เป็น L1 ที่สั้นอยู่แล้ว ข้อได้เปรียบด้านต้นทุนก็จะน้อย
3. ดูตัวอย่างที่แนวทาง (3) ตอบผิดด้านล่าง — ผิดแบบไหน แล้วควรเพิ่มข้อมูลเทรนแบบไหน?

In [ ]:
wrong = [x for x in rows[arm3] if x["score"] < 1]
print(f"แนวทาง (3) ตอบไม่ถูกทั้งข้อ {len(wrong)} จาก {len(rows[arm3])} ข้อ — แสดง 5 ข้อแรก\n")
for x in wrong[:5]:
    print(f"[{x['score']:.2f}] {x['text'][:50]}\n       ตอบ: {x['output'][:60]!r}\n")

## 4.5 บันทึก adapter
ไฟล์ LoRA แยกจากโมเดลหลัก ขนาดราว 14 MB (3.4 ล้านค่า × 4 ไบต์) — ถ้าจะแชร์ ต้องระบุ "Built with Llama" ตาม license ของ Llama 3.2

In [ ]:
model.save_pretrained("lora_adapter")
!du -sh lora_adapter        # ขนาดโฟลเดอร์ adapter (เทียบกับโมเดลเต็ม ≈ 2.5 GB)

## ✅ สรุป Lab 4
- ข้อมูล fine-tune = บทสนทนา (system + ข้อความ + คำตอบ) · label = ก้อนเดียวกันที่ตั้ง −100 (ไม่คิด loss) ทุกตำแหน่งยกเว้นคำตอบ · ใช้ system prompt เดิมตอนใช้งาน
- LoRA rank 16 ใน q, k, v, o เทรนเพียง ≈ 0.3% ของโมเดล แต่เปลี่ยนพฤติกรรมได้
- eval loss ที่ลดลงยังไม่ใช่หลักฐาน — ตัดสินด้วยชุด test ที่ไม่เคยใช้ปรับ prompt · 20 ข้อยังน้อย ดูค่า p ก่อนสรุป · คำตอบไม่จำเป็นต้องเป็น "fine-tune ดีกว่า" เสมอไป
- การบ้านใช้โค้ดชุดเดียวกันนี้กับงานของคุณเอง (HW_Starter_TH.ipynb)